# Exercise 9 — Network types: one encoder, many heads

Seven problems that look nothing alike — the fruit in a photo, the outline
of a skin lesion, the noise in an ECG, a noisy photo, a noisy ECG, the heartbeats in
an ECG, the folding of a protein — and seven networks that are almost the
same. In the first part we go through all seven together, as worked,
runnable examples. For each of them we ask the same six questions: what goes
**in**, what comes **out**, what is the **last layer**, which **loss**
trains it, which **metric** judges it, and what **baseline** it has to beat.
The second part applies the same recipe to seven new problems: you pick
three, write their specification, and an AI agent writes the code.

## When to use AI agents

- **Part 1: no agent.** Run the cells together with the teacher. At each of the six **TODOs**
  (in examples 1, 2, 3 and 6) write the missing lines yourself; the **Check** under it tells you
  whether they work. Then re-run that cell and the cells below it, because they use your result.
- **Part 2: use your coding agent.** You write the specification (input, output, last layer, loss, metric, baseline),
  the agent writes the code, and you check the results against the baseline and on the plots.

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit three tasks of Part 2 within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

In [ ]:
%matplotlib inline
import copy
import os
import shutil
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.signal import find_peaks
from skimage.filters import threshold_otsu
from sklearn.metrics import ConfusionMatrixDisplay
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Worker processes of the DataLoader read and augment the next batches while the network trains.
# On Windows and macOS, Jupyter workers cannot see a Dataset defined in the notebook: 0 there.
num_workers = 2 if sys.platform == "linux" else 0
print("torch", torch.__version__, "| device:", device)

## Part 1 — Worked examples: one encoder, many heads

The lecture on architectures ended with one table: the **encoder** — the
convolutions and poolings that turn the input into features — stays the
same, and only the **head** and the **loss** change with the task. Here is
that table for the seven examples of today:

| # | Example | Input → output | Last layer | Loss | Metric |
| --- | --- | --- | --- | --- | --- |
| 1 | Fruit type | photo → 1 of 3 classes | linear, on a pretrained MobileNet | cross-entropy | accuracy |
| 2 | Skin-lesion segmentation | photo → mask | 1 × 1 conv, 1 channel (sigmoid in the loss) | cross-entropy + Dice | Dice |
| 3 | ECG noise type | signal → 1 of 3 classes | linear, after global average pooling | cross-entropy | accuracy |
| 4 | Photo denoising | photo → photo | 1 × 1 conv, 3 channels, no activation | MSE | PSNR |
| 5 | ECG denoising | signal → signal | 1 × 1 conv, 1 channel, no activation | MSE | MSE |
| 6 | QRS detection | signal → heatmap → positions | 1 × 1 conv, 1 channel, no activation | MSE | F1 of the detected beats |
| 7 | Protein structure | sequence → 1 of 3 classes per residue | 1 × 1 conv, 3 channels | cross-entropy, padding ignored | accuracy per residue |

- **One answer per input** (rows 1, 3): the encoder's features are pooled into one vector, then a linear layer.
- **One answer per position** (rows 2, 4–7): a **U-Net** brings the features back to the input's size, then a 1 × 1 convolution.
- Signals and sequences (rows 3, 5–7) use `Conv1d`, photos (rows 1, 2, 4) `Conv2d`. Nothing else changes.

Every example is evaluated on its **test set**, which the training has never
seen, against a **baseline**: the simplest thing that could work without a
network (the most frequent class, doing nothing, a classical filter, a
threshold). A model that does not beat its baseline has learned nothing
useful, however low its training loss.

Examples 1 and 2 write everything out — the `Dataset` that reads the files,
the `DataLoader`, the training loop, the test evaluation — as in exercise 8.
After them the loop becomes a function `train()` for the other five.

### The data

One zip (about 7 MB) with seven small datasets, each stored differently, as
real data are: the label in the file name, in a second folder or in a CSV
table. Every example reads its files **from disk, one sample at a time**, in a
`Dataset` — the way to go when the data do not fit in memory. The cell
downloads the zip only when it is missing or out of date.

### Why a `Dataset`: we write one file read, PyTorch does the rest

`class FruitDataset(Dataset)` **inherits** from PyTorch's `Dataset`. We write
only the part that depends on how *our* data are stored, three methods:

- `__init__` — make the list of samples (file names, labels), but do not read them yet;
- `__len__` — how many samples there are;
- `__getitem__(i)` — read **sample `i`** from disk, augment it, return it as tensors.

Everything else is already written in the `DataLoader`: a new random order in
every epoch, cutting it into batches, stacking the samples into one tensor —
and **reading in parallel**. With `num_workers=2`, two worker processes call
`__getitem__` and prepare the next batches while the GPU trains on the current
one. This works without any code of ours, because `__getitem__(i)` needs
nothing but `i`: every sample can be read independently, by any process, in
any order. A `Dataset` is therefore a parallelization prepared in advance; we
only fill in how one file is read. Every example today has these three methods,
and only their content changes with the way the files are stored.

In [ ]:
URL = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/ex09_data.zip"
DATA_VERSION = "2026-11-06b"          # the version of the data this notebook was written for
DATA = Path("data/ex09")
if not (DATA / "VERSION").exists() or (DATA / "VERSION").read_text().strip() != DATA_VERSION:
    shutil.rmtree(DATA, ignore_errors=True)          # missing, or an older copy: download it again
    DATA.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, "data/ex09_data.zip")
    zipfile.ZipFile("data/ex09_data.zip").extractall("data")
assert (DATA / "VERSION").read_text().strip() == DATA_VERSION, "the downloaded data have another version"
SPLITS = ["train", "valid", "test"]
print(sorted(p.name for p in DATA.iterdir()))
results = {}   # the test result of every example, for the summary at the end

### The networks: one encoder, two kinds of head

**`Encoder`:** blocks of two 3 × 3 convolutions (each with batch norm and
ReLU), with a 2 × 2 max pooling between the blocks; `widths` is the number of
filters of each block. The same code builds a 1D (`Conv1d`) and a 2D
(`Conv2d`) encoder. It returns the output of *every* block, because the U-Net
needs them.

**`Classifier`:** the deepest features → the mean of every channel over all
positions (*global average pooling*, so any input size works) → a linear
layer, one score per class.

**`UNet`:** back up with transposed convolutions; at each resolution the
encoder's features are concatenated (the *skip connection*), and a final
1 × 1 convolution gives `c_out` numbers at every position.

Neither head has a softmax or a sigmoid: the losses apply it themselves, so
the networks output raw scores (*logits*).

The same principle as the `Dataset`: the networks **inherit** from
`nn.Module`, and we write only `__init__` (which layers) and `forward` (in what
order they are applied). Everything else is already in `nn.Module`:
`parameters()` for the optimizer, `.to(device)`, `train()` / `eval()`,
`state_dict()` for saving the best weights; the gradients come from autograd.

In [ ]:
def conv_block(dim, c_in, c_out):
    Conv = nn.Conv1d if dim == 1 else nn.Conv2d
    Norm = nn.BatchNorm1d if dim == 1 else nn.BatchNorm2d
    return nn.Sequential(Conv(c_in, c_out, 3, padding=1), Norm(c_out), nn.ReLU(),
                         Conv(c_out, c_out, 3, padding=1), Norm(c_out), nn.ReLU())


class Encoder(nn.Module):
    def __init__(self, dim, c_in, widths):
        super().__init__()
        self.blocks = nn.ModuleList()
        for w in widths:
            self.blocks.append(conv_block(dim, c_in, w))
            c_in = w                                    # the next block takes this block's output
        self.pool = nn.MaxPool1d(2) if dim == 1 else nn.MaxPool2d(2)

    def forward(self, x):
        features = []                                   # the output of every block, finest first
        for i, block in enumerate(self.blocks):
            if i > 0:
                x = self.pool(x)                        # halve the resolution between the blocks
            x = block(x)
            features.append(x)
        return features


class Classifier(nn.Module):
    def __init__(self, dim, c_in, n_classes, widths):
        super().__init__()
        self.encoder = Encoder(dim, c_in, widths)
        self.head = nn.Linear(widths[-1], n_classes)

    def forward(self, x):
        deepest = self.encoder(x)[-1]                      # (batch, channels, positions...)
        return self.head(deepest.flatten(2).mean(dim=2))   # global average pooling, then linear


class UNet(nn.Module):
    def __init__(self, dim, c_in, c_out, widths):
        super().__init__()
        self.encoder = Encoder(dim, c_in, widths)
        UpConv = nn.ConvTranspose1d if dim == 1 else nn.ConvTranspose2d
        Conv = nn.Conv1d if dim == 1 else nn.Conv2d
        self.up = nn.ModuleList()
        self.decode = nn.ModuleList()
        for i in reversed(range(len(widths) - 1)):     # from the deepest level up, e.g. 64 -> 32 -> 16 -> 8
            self.up.append(UpConv(widths[i + 1], widths[i], 2, stride=2))
            self.decode.append(conv_block(dim, 2 * widths[i], widths[i]))
        self.head = Conv(widths[0], c_out, 1)          # 1 x 1 convolution

    def forward(self, x):
        features = self.encoder(x)                     # finest first
        x = features.pop()                             # the deepest features
        for up, decode in zip(self.up, self.decode):
            skip = features.pop()                      # the encoder's features one level finer
            x = up(x, output_size=skip.shape[2:])      # double the resolution (exactly the skip's size)
            x = decode(torch.cat([skip, x], dim=1))    # skip connection by concatenation
        return self.head(x)


# the same building blocks, four different shapes out
for name, net, x in [
    ("1D classifier", Classifier(1, 1, 3, [16, 32, 64]), torch.zeros(2, 1, 256)),
    ("2D classifier", Classifier(2, 3, 3, [16, 32, 64]), torch.zeros(2, 3, 128, 96)),
    ("1D U-Net", UNet(1, 1, 1, [8, 16, 32, 64]), torch.zeros(2, 1, 256)),
    ("2D U-Net", UNet(2, 3, 1, [8, 16, 32, 64]), torch.zeros(2, 3, 128, 96)),
]:
    print(f"{name:14s} input {str(tuple(x.shape)):18s} -> output {tuple(net(x).shape)}")

### Example 1 — Fruit type (2D classification with a pretrained network)

Photos of apples, bananas and mandarins taken by students of this course,
downscaled to 128 rows × 96 columns (portrait): $128 \times 96 \times 3 = 36\,864$
numbers per photo, and fewer than 200 training photos — too few to learn
good features from scratch. So the encoder is **MobileNet**, pretrained
on ImageNet, as in exercise 8; only its last linear layer is replaced
by one with 3 outputs (*transfer learning*). It is the same idea as
`Classifier`: an encoder, global average pooling, a linear head.

| | Example 1 |
| --- | --- |
| input | one photo, shape `(3, 128, 96)` — 3 colour channels, 128 rows, 96 columns |
| output | 3 scores, shape `(3,)` |
| last layer | linear, replacing MobileNet's 1000-class layer |
| loss | cross-entropy |
| metric | accuracy on the test set |
| baseline | always answer the most frequent class of the training set |

**On disk:** one folder per split, and the class is the first part of the
file name: `fruits_classification/train/apple_012.jpg`, `.../banana_003.jpg`, …
`__init__` lists the files and their labels, `__getitem__` reads **one** photo
and returns it as a tensor `(3, 128, 96)` with its label.

**TODO 1:** in `__init__`, get the class index of every file from its name:
`apple_012.jpg` → `"apple"` → 0, its position in `fruit_classes`
(`Path.name`, `str.split`, `list.index`).

**Check:** the cell prints the photos per class, `[56 72 66]` for the training
set, and the titles in the figure match the fruit.

In [ ]:
fruit_classes = ["apple", "banana", "mandarin"]
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])   # ImageNet mean and std per channel


class FruitDataset(Dataset):
    def __init__(self, split, augment=False):
        self.files = sorted((DATA / "fruits_classification" / split).glob("*.jpg"))
        # The class index of every file: the class name is the part of the file name
        # before "_", the index is its position in fruit_classes.
        # TODO: your code here
        self.labels = [0] * len(self.files)  # PLACEHOLDER: replace
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        x = transforms.functional.to_tensor(img)       # (3, 128, 96), values in [0, 1]
        if self.augment and torch.rand(1) < 0.5:
            x = x.flip(-1)                             # mirror left-right
        return normalize(x), self.labels[i]


fruit_sets = {"train": FruitDataset("train", augment=True),
              "valid": FruitDataset("valid"),
              "test": FruitDataset("test")}
for s in SPLITS:
    print(f"{s:5s} {len(fruit_sets[s])} photos, per class {np.bincount(fruit_sets[s].labels)}")
x, label = fruit_sets["train"][0]
print("one sample:", tuple(x.shape), label)

fig, axes = plt.subplots(1, 8, figsize=(12, 2.6))
for ax, i in zip(axes, np.linspace(0, len(fruit_sets["train"]) - 1, 8).astype(int)):   # 8 photos spread over the sorted files: all three classes
    ax.imshow(Image.open(fruit_sets["train"].files[i]))
    ax.set_title(fruit_classes[fruit_sets["train"].labels[i]], fontsize=9)
    ax.axis("off")
plt.show()

MobileNet was trained on photos standardized with the ImageNet mean and
standard deviation of every channel, so ours are standardized the same
way (`normalize` in `__getitem__`). The training set mirrors a random half of
the photos left–right, every time a photo is read: a mirrored banana is still a
banana (*augmentation*, from the lecture). The validation and test sets are
not augmented.

The training loop is the one of exercise 8: in every epoch, training mode and
mini-batches with `zero_grad()`, loss, `backward()`, `step()`; then
evaluation mode and the validation loss without gradients. The weights of the
epoch with the lowest validation loss are kept — later epochs may overfit.

Both loaders get `num_workers`: the photos are read and mirrored by worker
processes while the GPU trains. With `num_workers=0` (the default) everything
runs in the main process, one after another, and the GPU waits for the disk.
Our photos are small, so here it saves little; with large images or heavy
augmentation it decides whether the GPU is busy or idle.

In [ ]:
train_loader = DataLoader(fruit_sets["train"], batch_size=16, shuffle=True, num_workers=num_workers)
valid_loader = DataLoader(fruit_sets["valid"], batch_size=16, num_workers=num_workers)

torch.manual_seed(0)
model = models.mobilenet_v3_small(weights="DEFAULT")                 # pretrained on ImageNet
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 3)   # the new head: 3 classes instead of 1000
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

epochs = 8
train_curve, valid_curve = [], []
best_loss = float("inf")
for epoch in range(epochs):
    model.train()
    losses = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    train_curve.append(np.mean(losses))

    model.eval()
    losses = []
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            losses.append(loss_fn(model(x), y).item())
    valid_curve.append(np.mean(losses))

    if valid_curve[-1] < best_loss:                       # the best epoch so far: keep its weights
        best_loss, best_epoch = valid_curve[-1], epoch + 1
        best_state = copy.deepcopy(model.state_dict())
    print(f"epoch {epoch + 1:2d}   training loss {train_curve[-1]:.4f}   validation loss {valid_curve[-1]:.4f}")

model.load_state_dict(best_state)
fruit_net = model
print(f"best validation loss {best_loss:.4f} in epoch {best_epoch}")

plt.figure(figsize=(5, 2.5))
plt.plot(range(1, epochs + 1), train_curve, label="training")
plt.plot(range(1, epochs + 1), valid_curve, label="validation")
plt.axvline(best_epoch, color="gray", linestyle=":")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

**TODO 2:** the test loop below collects the predicted classes `pred` and
the true classes `y_test` of all test photos; compute the **test accuracy**
from them.

**Check:** the figure marks the wrong predictions in red, so the accuracy
must be $1 - (\text{number of red titles}) / 21$.

In [ ]:
test_loader = DataLoader(fruit_sets["test"], batch_size=16)
fruit_net.eval()
pred, y_test = [], []
with torch.no_grad():
    for x, y in test_loader:
        scores = fruit_net(x.to(device))
        pred.append(scores.argmax(dim=1).cpu())
        y_test.append(y)
pred, y_test = torch.cat(pred), torch.cat(y_test)

# The test accuracy: the fraction of test photos whose predicted class is the true class.
# TODO: your code here
accuracy = 0.0  # PLACEHOLDER: replace
majority = np.bincount(fruit_sets["train"].labels).argmax()
baseline = (y_test == majority).float().mean().item()
print(f"test accuracy {accuracy:.3f}   baseline (always '{fruit_classes[majority]}') {baseline:.3f}")
results["1 Fruit type"] = dict(metric="accuracy", model=float(accuracy), baseline=baseline)

fig, axes = plt.subplots(3, 7, figsize=(10, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(Image.open(fruit_sets["test"].files[i]))
    ax.set_title(fruit_classes[pred[i]], fontsize=8, color="black" if pred[i] == y_test[i] else "red")
    ax.axis("off")
plt.suptitle("test photos with the predicted class (red = wrong)")
plt.show()

### Example 2 — Skin-lesion segmentation (2D, a label for every pixel)

Dermoscopy photos (a magnified, lit photo of the skin) of pigmented skin
lesions — moles and melanomas — each with a mask drawn by an expert: 1 where
the lesion is, 0 elsewhere. Outlining the lesion is the first step of an
automatic melanoma screening. The answer is no longer one class per photo,
but one class per **pixel** — the output has the size of the input. That is
the U-Net. The data: 150 photos of the ISIC 2016 challenge (licence CC-0),
split 100 / 20 / 30 and squashed to 96 rows × 128 columns (landscape).

With two classes (lesion, background) one output channel is enough: its
sigmoid is the probability of "lesion" — the same as a softmax over two
channels. The loss is the sum the lecture recommends: the per-pixel
**cross-entropy** plus the **Dice loss** $1 - \text{Dice}$, which looks at
the overlap of the whole mask and therefore does not let the background
(most of the pixels) dominate.

| | Example 2 |
| --- | --- |
| input | one photo, shape `(3, 96, 128)` — 96 rows, 128 columns |
| output | 1 score per pixel, shape `(1, 96, 128)` |
| last layer | 1 × 1 convolution, 1 channel (the sigmoid is in the loss) |
| loss | binary cross-entropy + Dice loss |
| metric | Dice of the thresholded mask, per test photo: mean and median |
| baseline | all background (Dice 0); Otsu's threshold on one colour channel, the channel picked on the validation set |

**On disk:** every split has two folders, the photos in `images/` and the
masks in `masks/`, with the same file name: `skin_lesions/train/images/007.jpg`
and `skin_lesions/train/masks/007.png`.

**TODO 3:** in `__getitem__`, find the mask of photo `i`: the same file
name with the extension `.png`, in the folder `masks` next to `images`
(`Path.parent`, `Path.stem`).

**Check:** the cell prints the fraction of lesion pixels (28.7 % in the
training set), and the lower row of the first figure shows white lesions on black.

**TODO 4:** a dermoscopy photo has no "up", so the training set flips at
random: left–right with probability 1/2 and, independently, up–down with
probability 1/2. One `torch.rand(1)` draw per flip, applied to the photo
**and** its mask, or the mask no longer fits the photo (`tensor.flip(-1)`
flips the columns, `flip(-2)` the rows).

**Check:** in the second figure one training photo is read six times — flipped
differently, with the red outline of its mask always on the lesion.

In [ ]:
class LesionDataset(Dataset):
    def __init__(self, split, augment=False):
        self.files = sorted((DATA / "skin_lesions" / split / "images").glob("*.jpg"))
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        # The path of the mask of photo i: the same file name with the extension .png, in the folder masks.
        # TODO: your code here
        mask_path = self.files[i]  # PLACEHOLDER: replace
        img = Image.open(self.files[i]).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        x = transforms.functional.to_tensor(img)       # (3, 96, 128), values in [0, 1]
        y = transforms.functional.to_tensor(mask)      # (1, 96, 128), 0 = skin, 1 = lesion
        if self.augment:
            # With probability 1/2 flip left-right, and independently with probability 1/2 up-down;
            # one random draw (torch.rand) for each flip, applied to the photo and its mask together.
            # TODO: your code here
            ...
        return x, y


lesion_sets = {"train": LesionDataset("train", augment=True),
               "valid": LesionDataset("valid"),
               "test": LesionDataset("test")}
for s in SPLITS:
    lesion_fraction = np.mean([y.mean().item() for _, y in lesion_sets[s]])
    print(f"{s:5s} {len(lesion_sets[s])} photos, lesion pixels {lesion_fraction:.1%}")

fig, axes = plt.subplots(2, 6, figsize=(12, 3.2))
for k in range(6):
    x, y = lesion_sets["valid"][k]
    axes[0, k].imshow(x.permute(1, 2, 0))
    axes[1, k].imshow(y[0], cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

# training photo 80 (its lesion is off-centre) read six times: the flips are random, the outline of the mask (red) stays on the lesion
fig, axes = plt.subplots(1, 6, figsize=(12, 1.7))
for ax in axes:
    x, y = lesion_sets["train"][80]
    ax.imshow(x.permute(1, 2, 0))
    ax.contour(y[0], levels=[0.5], colors="red", linewidths=1)
    ax.axis("off")
plt.show()

The loop is the one of example 1; only the data, the network and the loss
differ. The loss is written out in both loops: the sigmoid of the logits is
the probability of "lesion", the Dice of the batch is computed from it (+1 in
the numerator and the denominator, so that an empty mask does not divide by
zero), and the loss is the binary cross-entropy plus $1 - \text{Dice}$.

In [ ]:
train_loader = DataLoader(lesion_sets["train"], batch_size=8, shuffle=True, num_workers=num_workers)
valid_loader = DataLoader(lesion_sets["valid"], batch_size=8, num_workers=num_workers)

torch.manual_seed(0)
model = UNet(dim=2, c_in=3, c_out=1, widths=[8, 16, 32, 64])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 25
train_curve, valid_curve = [], []
best_loss = float("inf")
for epoch in range(epochs):
    model.train()
    losses = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        p = torch.sigmoid(logits)                                  # the probability of "lesion"
        dice = (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
        loss = F.binary_cross_entropy_with_logits(logits, y) + 1 - dice
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    train_curve.append(np.mean(losses))

    model.eval()
    losses = []
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            p = torch.sigmoid(logits)
            dice = (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
            loss = F.binary_cross_entropy_with_logits(logits, y) + 1 - dice
            losses.append(loss.item())
    valid_curve.append(np.mean(losses))

    if valid_curve[-1] < best_loss:                       # the best epoch so far: keep its weights
        best_loss, best_epoch = valid_curve[-1], epoch + 1
        best_state = copy.deepcopy(model.state_dict())
    print(f"epoch {epoch + 1:2d}   training loss {train_curve[-1]:.4f}   validation loss {valid_curve[-1]:.4f}")

model.load_state_dict(best_state)
seg_net = model
print(f"best validation loss {best_loss:.4f} in epoch {best_epoch}")

plt.figure(figsize=(5, 2.5))
plt.plot(range(1, epochs + 1), train_curve, label="training")
plt.plot(range(1, epochs + 1), valid_curve, label="validation")
plt.axvline(best_epoch, color="gray", linestyle=":")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

**The baselines.**

- **All background:** a pixel accuracy of about 65 % on the test photos — and
  a Dice of 0, which is why pixel accuracy is not the metric.
- **Otsu:** a lesion is darker than the skin around it, so threshold every
  photo at its own Otsu level (the threshold that best separates the histogram
  into two groups) and keep the **dark** side. Which channel — red, green,
  blue or their mean (grey) — is chosen on the **validation** set, like every
  tuned parameter of today.

In [ ]:
# Otsu on the validation set: which channel gives the best mean Dice?
best_dice = 0
for channel in ["R", "G", "B", "grey"]:
    dices = []
    for x, y in lesion_sets["valid"]:
        c = x.mean(dim=0) if channel == "grey" else x["RGB".index(channel)]   # one channel, (96, 128)
        mask = (c < threshold_otsu(c.numpy())).float()                         # the dark side of the threshold
        dices.append((2 * (mask * y[0]).sum() / (mask.sum() + y.sum())).item())
    print(f"Otsu on {channel:4s}: validation Dice {np.mean(dices):.2f}")
    if np.mean(dices) > best_dice:
        best_dice, best_channel = np.mean(dices), channel

# the test set: the U-Net and Otsu on the chosen channel
test_images = torch.stack([x for x, _ in lesion_sets["test"]])     # (30, 3, 96, 128)
test_masks = torch.stack([y for _, y in lesion_sets["test"]])      # (30, 1, 96, 128)
seg_net.eval()
with torch.no_grad():
    net_masks = (seg_net(test_images.to(device)).cpu() > 0).float()   # logit > 0 is sigmoid > 0.5
otsu_masks = []
for x in test_images:
    c = x.mean(dim=0) if best_channel == "grey" else x["RGB".index(best_channel)]
    otsu_masks.append((c < threshold_otsu(c.numpy())).float()[None])
otsu_masks = torch.stack(otsu_masks)

# the Dice of every test photo
net_dice, otsu_dice = [], []
for i in range(len(test_masks)):
    y = test_masks[i]
    net_dice.append((2 * (net_masks[i] * y).sum() / (net_masks[i].sum() + y.sum())).item())
    otsu_dice.append((2 * (otsu_masks[i] * y).sum() / (otsu_masks[i].sum() + y.sum())).item())
net_dice, otsu_dice = np.array(net_dice), np.array(otsu_dice)

print(f"all background: pixel accuracy {1 - test_masks.mean().item():.3f}, Dice 0")
print(f"test Dice (30 photos)   Otsu {best_channel}: mean {otsu_dice.mean():.3f}, median {np.median(otsu_dice):.3f}   "
      f"U-Net: mean {net_dice.mean():.3f}, median {np.median(net_dice):.3f}")
results["2 Skin-lesion segmentation"] = dict(metric="Dice", model=net_dice.mean(), baseline=otsu_dice.mean())

order = np.argsort(otsu_dice)
show = [*order[:4], order[len(order) // 2], order[-1]]   # the four worst photos for Otsu, a median one, the best one
fig, axes = plt.subplots(len(show), 4, figsize=(8, 1.6 * len(show)))
for row, i in zip(axes, show):
    row[0].imshow(test_images[i].permute(1, 2, 0))
    row[0].set_title(f"test photo {i}", fontsize=9)
    row[1].imshow(test_masks[i, 0], cmap="gray")
    row[1].set_title("true mask", fontsize=9)
    row[2].imshow(otsu_masks[i, 0], cmap="gray")
    row[2].set_title(f"Otsu {otsu_dice[i]:.2f}", fontsize=9)
    row[3].imshow(net_masks[i, 0], cmap="gray")
    row[3].set_title(f"U-Net {net_dice[i]:.2f}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

### From now on the loop is a function

You have now seen the same loop twice; only the data, the network and the
loss changed. From now on it is a function: `train()` takes exactly these
three (and the number of epochs, the learning rate, the batch size and the number of workers), keeps
the weights of the best validation epoch and draws the loss curves.
`predict()` returns all outputs of a model for a dataset, with the targets.
Augmentation is not their business — it lives in the `Dataset`.

In [ ]:
def train(model, train_set, val_set, loss_fn, epochs, lr=1e-3, batch_size=16, num_workers=num_workers):
    """Adam on mini-batches; returns the model with the weights of its best validation epoch."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    valid_loader = DataLoader(val_set, batch_size=batch_size, num_workers=num_workers)
    train_curve, valid_curve = [], []
    best_loss = float("inf")
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        losses = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        train_curve.append(np.mean(losses))

        model.eval()
        losses = []
        with torch.no_grad():
            for x, y in valid_loader:
                x, y = x.to(device), y.to(device)
                losses.append(loss_fn(model(x), y).item())
        valid_curve.append(np.mean(losses))

        if valid_curve[-1] < best_loss:
            best_loss, best_epoch = valid_curve[-1], epoch + 1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"{epochs} epochs in {time.perf_counter() - start:.0f} s; best validation loss {best_loss:.4f} in epoch {best_epoch}")
    plt.figure(figsize=(5, 2.5))
    plt.plot(range(1, epochs + 1), train_curve, label="training")
    plt.plot(range(1, epochs + 1), valid_curve, label="validation")
    plt.axvline(best_epoch, color="gray", linestyle=":")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.show()
    return model


def predict(model, dataset, batch_size=64):
    """All outputs of the model for a dataset (on the CPU), and the targets."""
    model.eval()
    outputs, targets = [], []
    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=batch_size):
            outputs.append(model(x.to(device)).cpu())
            targets.append(y)
    return torch.cat(outputs), torch.cat(targets)

### Example 3 — ECG noise type (1D classification)

One-lead ECG signals of 256 samples. Each is clean, or has Gaussian noise
(every sample shifted a little), or salt-and-pepper noise (a few samples
replaced by large spikes). The network says which.

| | Example 3 |
| --- | --- |
| input | one signal, shape `(1, 256)` — 1 channel, 256 samples |
| output | 3 scores, shape `(3,)` |
| last layer | linear, after global average pooling |
| loss | cross-entropy |
| metric | accuracy on the test set |
| baseline | always answer the most frequent class of the training set |

**On disk:** all signals are in one folder, `ecg_noise_class/signals/00000.npy`
… `00917.npy`, and one table `ecg_noise_class/labels.csv` for the whole
dataset says, for every file, its split and its class:

```
file,split,label
signals/00000.npy,train,clean
...
```

**TODO 5:** in `__init__`, read `labels.csv` with `pd.read_csv` and keep
only the rows of the requested `split`.

**Check:** the cell prints 747 / 85 / 86 signals in the three splits, the
training set `[261 236 250]` per class. Until TODO 5 is done the cell stops
with "TODO 5 is not done yet".

In [ ]:
ecg_classes = ["clean", "gaussian", "saltAndPepper"]


class EcgNoiseDataset(Dataset):
    def __init__(self, split):
        # The table labels.csv as a DataFrame, only the rows of this split.
        # TODO: your code here
        table = pd.DataFrame(columns=["file", "split", "label"])  # PLACEHOLDER: replace
        self.files = [DATA / "ecg_noise_class" / file for file in table["file"]]   # the paths are relative to the table
        self.labels = [ecg_classes.index(name) for name in table["label"]]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        signal = torch.tensor(np.load(self.files[i]), dtype=torch.float32)   # (256,), stored as float16
        return signal[None], self.labels[i]                                   # (1, 256): one channel


noise_sets = {s: EcgNoiseDataset(s) for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s} {len(noise_sets[s])} signals, per class {np.bincount(noise_sets[s].labels)}")
assert len(noise_sets["train"]) > 0, "TODO 5 is not done yet: no signals were read"

fig, axes = plt.subplots(1, 3, figsize=(12, 2.5), sharey=True)
for label, ax in enumerate(axes):
    i = noise_sets["train"].labels.index(label)        # the first training signal of this class
    signal, _ = noise_sets["train"][i]
    ax.plot(signal[0])
    ax.set_title(ecg_classes[label])
plt.show()

A small 1D classifier: three blocks with 16, 32 and 64 filters, so the
deepest features have 64 samples (256 / 4) of 64 channels; their averages
go into a linear layer with 3 outputs.

In [ ]:
torch.manual_seed(0)
noise_net = train(Classifier(dim=1, c_in=1, n_classes=3, widths=[16, 32, 64]),
                  noise_sets["train"], noise_sets["valid"], nn.CrossEntropyLoss(), epochs=10, batch_size=32)

In [ ]:
scores, y_test = predict(noise_net, noise_sets["test"])
pred = scores.argmax(dim=1)
accuracy = (pred == y_test).float().mean().item()
majority = np.bincount(noise_sets["train"].labels).argmax()
baseline = (y_test == majority).float().mean().item()
print(f"test accuracy {accuracy:.3f}   baseline (always '{ecg_classes[majority]}') {baseline:.3f}")
results["3 ECG noise type"] = dict(metric="accuracy", model=accuracy, baseline=baseline)

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=ecg_classes, colorbar=False)
plt.show()
print("misclassified test signals:", np.flatnonzero(pred != y_test))

### Example 4 — Photo denoising (2D image-to-image regression)

Fruit photos with Gaussian noise: every pixel and channel gets a random
number with standard deviation $\sigma = 0.12$ added (on the scale 0–1).
The output is again a photo — three **real numbers** per pixel, not classes.
So the last layer has **no activation** and the loss is the **mean squared
error**.

**`Residual`** (the cell below) wraps the U-Net, following the lecture: the
network learns the *noise* $f_\theta(\mathbf x)$ and the output is
$\hat{\mathbf y} = \mathbf x - f_\theta(\mathbf x)$. If the predicted noise
is zero, the output equals the input.

The metric is the **PSNR** (peak signal-to-noise ratio),
$10 \log_{10}(1 / \text{MSE})$ in decibels for images with values in
$[0, 1]$: higher is better, +3 dB means half the mean squared error.

| | Example 4 |
| --- | --- |
| input | a noisy photo, shape `(3, 128, 96)`; 64 × 64 crops in training |
| output | the clean photo, shape `(3, 128, 96)` |
| last layer | 1 × 1 convolution, 3 channels, no activation (inside `Residual`) |
| loss | MSE |
| metric | PSNR on the test photos |
| baseline | do nothing (the noisy photo) or a Gaussian blur, chosen (with the blur's sigma) on the validation set |

**On disk:** clean photos only, `fruits_denoising/<split>/000.jpg`; the noise
is added in `__getitem__`, when a photo is read:

- **Training photos** (`augment=True`): a random 64 × 64 **crop** of a randomly
  mirrored photo, with **new noise** every time it is read (noise is free).
  The U-Net is convolutional, so afterwards it takes the whole photo.
- **Validation and test photos:** whole, with the **same** noise in every run
  (a random generator seeded with the index of the photo), so the metric is
  reproducible.

In [ ]:
class Residual(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):
        return x - self.net(x)              # the network predicts the noise

In [ ]:
NOISE = 0.12                                 # standard deviation of the noise, on the scale 0-1
CROP = 64


class NoisyPhotoDataset(Dataset):
    def __init__(self, split, augment=False):
        self.files = sorted((DATA / "fruits_denoising" / split).glob("*.jpg"))
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        clean = transforms.functional.to_tensor(Image.open(self.files[i]).convert("RGB"))   # (3, 128, 96)
        if self.augment:                               # a random mirror, a random crop, new noise every time
            if torch.rand(1) < 0.5:
                clean = clean.flip(-1)
            r = torch.randint(0, clean.shape[1] - CROP + 1, (1,)).item()
            c = torch.randint(0, clean.shape[2] - CROP + 1, (1,)).item()
            clean = clean[:, r:r + CROP, c:c + CROP]
            noise = torch.randn(clean.shape)
        else:                                          # the whole photo, the same noise in every run
            noise = torch.randn(clean.shape, generator=torch.Generator().manual_seed(i))
        return (clean + NOISE * noise).clamp(0, 1), clean


photo_sets = {"train": NoisyPhotoDataset("train", augment=True),
              "valid": NoisyPhotoDataset("valid"),
              "test": NoisyPhotoDataset("test")}
for s in SPLITS:
    x, y = photo_sets[s][0]
    print(f"{s:5s} {len(photo_sets[s])} photos, one sample: noisy {tuple(x.shape)}, clean {tuple(y.shape)}")

torch.manual_seed(0)
photo_net = train(Residual(UNet(dim=2, c_in=3, c_out=3, widths=[16, 32, 64])),
                  photo_sets["train"], photo_sets["valid"], nn.MSELoss(), epochs=40, lr=2e-3, batch_size=8)

In [ ]:
# the baseline on the validation set: do nothing (sigma 0), or a Gaussian blur of which sigma?
noisy_valid = torch.stack([x for x, _ in photo_sets["valid"]])
clean_valid = torch.stack([y for _, y in photo_sets["valid"]])
best_psnr = 10 * np.log10(1 / F.mse_loss(noisy_valid, clean_valid).item())
best_sigma = 0
print(f"do nothing:            validation PSNR {best_psnr:.2f} dB")
for sigma in [0.5, 0.75, 1.0, 1.5, 2.0]:
    blurred = torch.tensor(gaussian_filter(noisy_valid.numpy(), sigma=(0, 0, sigma, sigma)))   # blur rows and columns only
    psnr_valid = 10 * np.log10(1 / F.mse_loss(blurred, clean_valid).item())
    print(f"Gaussian blur, sigma {sigma}: validation PSNR {psnr_valid:.2f} dB")
    if psnr_valid > best_psnr:
        best_psnr, best_sigma = psnr_valid, sigma

# the test set
denoised, clean_test = predict(photo_net, photo_sets["test"])
noisy_test = torch.stack([x for x, _ in photo_sets["test"]])
if best_sigma == 0:
    baseline_test = noisy_test
else:
    baseline_test = torch.tensor(gaussian_filter(noisy_test.numpy(), sigma=(0, 0, best_sigma, best_sigma)))
psnr_noisy = 10 * np.log10(1 / F.mse_loss(noisy_test, clean_test).item())
psnr_baseline = 10 * np.log10(1 / F.mse_loss(baseline_test, clean_test).item())
psnr_net = 10 * np.log10(1 / F.mse_loss(denoised, clean_test).item())
print(f"test PSNR   noisy {psnr_noisy:.1f} dB   baseline (blur sigma {best_sigma}) {psnr_baseline:.1f} dB   U-Net {psnr_net:.1f} dB")
results["4 Photo denoising"] = dict(metric="PSNR [dB]", model=psnr_net, baseline=psnr_baseline)

fig, axes = plt.subplots(2, 4, figsize=(9, 6.5))
for row, i in zip(axes, [0, 12]):
    row[0].imshow(noisy_test[i].permute(1, 2, 0))
    row[0].set_title("noisy", fontsize=9)
    row[1].imshow(baseline_test[i].permute(1, 2, 0))
    row[1].set_title(f"Gaussian blur (sigma {best_sigma})", fontsize=9)
    row[2].imshow(denoised[i].clamp(0, 1).permute(1, 2, 0))
    row[2].set_title("U-Net", fontsize=9)
    row[3].imshow(clean_test[i].permute(1, 2, 0))
    row[3].set_title("clean", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.show()

### Example 5 — ECG denoising (1D signal-to-signal regression)

Example 4 in 1D: pairs of a noisy and a clean ECG signal of 256 samples.
The same `Residual(UNet(...))`, only with `dim=1`.

| | Example 5 |
| --- | --- |
| input | a noisy signal, shape `(1, 256)` |
| output | the clean signal, shape `(1, 256)` |
| last layer | 1 × 1 convolution, 1 channel, no activation (inside `Residual`) |
| loss | MSE |
| metric | MSE on the test signals |
| baseline | do nothing or a Gaussian smoothing, chosen (with the smoothing's sigma) on the validation set |

**On disk:** pairs `ecg_denoising/<split>/00012_noisy.npy` and `00012_clean.npy`.

In [ ]:
class EcgDenoisingDataset(Dataset):
    def __init__(self, split):
        self.files = sorted((DATA / "ecg_denoising" / split).glob("*_noisy.npy"))   # each with its _clean.npy

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        noisy = torch.tensor(np.load(self.files[i]), dtype=torch.float32)
        clean = torch.tensor(np.load(str(self.files[i]).replace("_noisy", "_clean")), dtype=torch.float32)
        return noisy[None], clean[None]                                    # (1, 256) each


ecg_sets = {s: EcgDenoisingDataset(s) for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s} {len(ecg_sets[s])} pairs, one signal {tuple(ecg_sets[s][0][0].shape)}")

torch.manual_seed(0)
ecg_net = train(Residual(UNet(dim=1, c_in=1, c_out=1, widths=[16, 32, 64, 128])),
                ecg_sets["train"], ecg_sets["valid"], nn.MSELoss(), epochs=20)

In [ ]:
# the baseline on the validation set: do nothing (sigma 0), or a Gaussian smoothing of which sigma?
noisy_valid = torch.stack([x for x, _ in ecg_sets["valid"]])
clean_valid = torch.stack([y for _, y in ecg_sets["valid"]])
best_mse = F.mse_loss(noisy_valid, clean_valid).item()
best_sigma = 0
print(f"do nothing:                 validation MSE {best_mse:.5f}")
for sigma in [0.5, 0.75, 1.0, 1.5, 2.0]:
    smoothed = torch.tensor(gaussian_filter1d(noisy_valid.numpy(), sigma, axis=-1))
    mse = F.mse_loss(smoothed, clean_valid).item()
    print(f"Gaussian smoothing, sigma {sigma}: validation MSE {mse:.5f}")
    if mse < best_mse:
        best_mse, best_sigma = mse, sigma

# the test set
denoised, clean_test = predict(ecg_net, ecg_sets["test"])
noisy_test = torch.stack([x for x, _ in ecg_sets["test"]])
if best_sigma == 0:
    baseline_test = noisy_test
else:
    baseline_test = torch.tensor(gaussian_filter1d(noisy_test.numpy(), best_sigma, axis=-1))
mse_noisy = F.mse_loss(noisy_test, clean_test).item()
mse_baseline = F.mse_loss(baseline_test, clean_test).item()
mse_net = F.mse_loss(denoised, clean_test).item()
print(f"test MSE   noisy {mse_noisy:.5f}   baseline (smoothing sigma {best_sigma}) {mse_baseline:.5f}   U-Net {mse_net:.5f}")
results["5 ECG denoising"] = dict(metric="MSE", model=mse_net, baseline=mse_baseline)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, i in zip(axes, [0, 5]):
    ax.plot(noisy_test[i, 0], color="lightgray", label="noisy")
    ax.plot(baseline_test[i, 0], label=f"Gaussian smoothing (sigma {best_sigma})")
    ax.plot(denoised[i, 0], label="U-Net")
    ax.plot(clean_test[i, 0], "k--", linewidth=1, label="clean")
axes[0].legend(ncol=4, fontsize=8)
axes[1].set_xlabel("sample")
plt.show()

### Example 6 — QRS detection (1D dense prediction: a heatmap, then peaks)

ECG signals of 1280 samples (about five heartbeats); the target is the position
of every **QRS complex** (the tall spike of a heartbeat). Positions are a
list of variable length — not something a network outputs directly. The
trick: turn the positions into a **heatmap**, a signal with a Gaussian bump
of height 1 at every QRS and 0 elsewhere. Predicting it is
signal-to-signal regression (example 5 again, without `Residual`); the
positions are then the peaks of the predicted heatmap
(`scipy.signal.find_peaks`).

A detected peak counts as correct (*TP*) if it is at most 25 samples from a
true QRS that no other peak has claimed; the rest are false alarms (*FP*),
and the unclaimed QRS are misses (*FN*). The metric is
$F_1 = 2\,TP / (2\,TP + FP + FN)$, the $F_1$ score of the second lecture.

| | Example 6 |
| --- | --- |
| input | a signal, shape `(1, 1280)` |
| output | a heatmap, shape `(1, 1280)` → a list of QRS positions |
| last layer | 1 × 1 convolution, 1 channel, no activation |
| loss | MSE against the heatmap |
| metric | F1 of the detected QRS (tolerance ±25 samples) on the test signals |
| baseline | a classical detector: peaks of the smoothed squared derivative, threshold tuned on the validation set |

**On disk:** the signals are `ecg_qrs/<split>/00012_ecg.npy` (the number is
the record), and the annotations are one table `ecg_qrs/qrs.csv` with a row
for every QRS complex — the record and the position (sample index) — so a
record has several rows. `__init__` (given) collects the positions of every
record into the list `self.qrs`.

**TODO 6:** in `__getitem__`, build the target heatmap of record `i`: zeros as
long as the ECG, a 1 at each position in `self.qrs[i]`, then
`gaussian_filter1d(..., WIDTH, mode="constant")` (`"constant"`: a QRS near the
end is not reflected into a second bump). The filter's weights sum to 1, so
multiply by `WIDTH * np.sqrt(2 * np.pi)` to make every bump 1 high.

**Check:** the cell prints the maximum of the heatmap (1.000) and the number
of its bumps, which equals the number of QRS of the record; in the plot every
bump sits on a QRS spike.

In [ ]:
WIDTH = 8                                    # standard deviation of a bump, in samples


class QrsDataset(Dataset):
    def __init__(self, split):
        self.files = sorted((DATA / "ecg_qrs" / split).glob("*_ecg.npy"))
        table = pd.read_csv(DATA / "ecg_qrs" / "qrs.csv")                   # columns record, position
        records = [int(f.name.split("_")[0]) for f in self.files]           # 00012_ecg.npy -> 12
        # the QRS positions of every record (also for the F1 at the end: a list of positions is not a tensor)
        self.qrs = [table.loc[table["record"] == r, "position"].to_numpy() for r in records]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        ecg = torch.tensor(np.load(self.files[i]), dtype=torch.float32)
        # The heatmap: zeros of the length of the ECG with a 1 at every QRS position of record i,
        # smoothed with gaussian_filter1d (standard deviation WIDTH, mode "constant") and scaled to height 1.
        # TODO: your code here
        heat = np.zeros(len(ecg))  # PLACEHOLDER: replace
        return ecg[None], torch.tensor(heat, dtype=torch.float32)[None]   # (1, 1280) each


qrs_sets = {s: QrsDataset(s) for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s} {len(qrs_sets[s])} records of {qrs_sets[s][0][0].shape[-1]} samples, "
          f"{np.mean([len(t) for t in qrs_sets[s].qrs]):.1f} QRS per record")

x, heat = qrs_sets["train"][0]
print(f"record 0: heatmap maximum {heat.max():.3f}, {len(find_peaks(heat[0], height=0.5)[0])} bumps, "
      f"{len(qrs_sets['train'].qrs[0])} QRS in qrs.csv")
plt.figure(figsize=(12, 2.5))
plt.plot(x[0], label="ECG")
plt.plot(0.2 * heat[0], label="target heatmap (scaled by 0.2)")
plt.legend(fontsize=8)
plt.show()

In [ ]:
torch.manual_seed(0)
qrs_net = train(UNet(dim=1, c_in=1, c_out=1, widths=[8, 16, 32, 64]),
                qrs_sets["train"], qrs_sets["valid"], nn.MSELoss(), epochs=15)

In [ ]:
def f1_detections(true_list, found_list, tolerance=25):
    """F1 over all records: a found peak at most `tolerance` samples from a still unclaimed true QRS is a TP."""
    tp = fp = fn = 0
    for true, found in zip(true_list, found_list):
        free = list(found)
        for t in true:
            distances = [abs(t - f) for f in free]
            if distances and min(distances) <= tolerance:
                free.pop(int(np.argmin(distances)))
                tp += 1
            else:
                fn += 1
        fp += len(free)
    return 2 * tp / (2 * tp + fp + fn)


# the U-Net's curves: the predicted heatmaps, (records, 1280)
heat_valid = predict(qrs_net, qrs_sets["valid"])[0][:, 0].numpy()
heat_test = predict(qrs_net, qrs_sets["test"])[0][:, 0].numpy()

# the classical detector's curves: the squared derivative, smoothed, scaled to a maximum of 1 in every record
# (np.diff is one sample shorter: the positions shift by at most one sample)
signals_valid = torch.stack([x for x, _ in qrs_sets["valid"]])[:, 0].numpy()
signals_test = torch.stack([x for x, _ in qrs_sets["test"]])[:, 0].numpy()
energy_valid = gaussian_filter1d(np.diff(signals_valid, axis=-1) ** 2, 3, axis=-1)
energy_valid = energy_valid / energy_valid.max(axis=-1, keepdims=True)
energy_test = gaussian_filter1d(np.diff(signals_test, axis=-1) ** 2, 3, axis=-1)
energy_test = energy_test / energy_test.max(axis=-1, keepdims=True)

# the peak threshold of each detector is chosen on the validation set;
# peaks at least 50 samples apart (the closest two QRS in the data are 52 samples apart)
best_net = best_classic = -1
for height in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    found_net = [find_peaks(curve, height=height, distance=50)[0] for curve in heat_valid]
    found_classic = [find_peaks(curve, height=height, distance=50)[0] for curve in energy_valid]
    f1_net = f1_detections(qrs_sets["valid"].qrs, found_net)
    f1_classic = f1_detections(qrs_sets["valid"].qrs, found_classic)
    print(f"threshold {height}: validation F1   classical {f1_classic:.3f}   U-Net {f1_net:.3f}")
    if f1_net > best_net:
        best_net, h_net = f1_net, height
    if f1_classic > best_classic:
        best_classic, h_classic = f1_classic, height

# the test set
found_net = [find_peaks(curve, height=h_net, distance=50)[0] for curve in heat_test]
found_classic = [find_peaks(curve, height=h_classic, distance=50)[0] for curve in energy_test]
f1_net = f1_detections(qrs_sets["test"].qrs, found_net)
f1_classic = f1_detections(qrs_sets["test"].qrs, found_classic)
print(f"test F1   classical detector (threshold {h_classic}) {f1_classic:.3f}   U-Net (threshold {h_net}) {f1_net:.3f}")
results["6 QRS detection"] = dict(metric="F1", model=f1_net, baseline=f1_classic)

f1_per_record = [f1_detections([true], [found]) for true, found in zip(qrs_sets["test"].qrs, found_net)]
worst = np.argsort(f1_per_record)[:2]                  # the two test records with the worst F1 of the U-Net
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, i in zip(axes, [0, *worst]):
    true = qrs_sets["test"].qrs[i]
    ax.plot(signals_test[i] / np.abs(signals_test[i]).max(), color="gray", label="ECG (scaled)")
    ax.plot(heat_test[i], label="predicted heatmap")
    ax.plot(true, np.full(len(true), 1.15), "kv", label="true QRS")
    ax.plot(found_net[i], np.full(len(found_net[i]), 1.3), "rv", label="detected")
    ax.set_title(f"test record {i}", fontsize=9)
axes[0].legend(ncol=4, fontsize=8)
axes[2].set_xlabel("sample")
plt.show()

### Example 7 — Protein secondary structure (1D, a class for every residue)

A protein is a chain of amino acids (*residues*), 20 kinds plus `X` for an
unknown one. Locally the chain folds into a helix (`H`), a sheet strand
(`E`), or neither (coil, `C`). The task: predict the fold of every residue
from the sequence alone (data: CullPDB, a standard protein benchmark). This
is segmentation in 1D: the input is the sequence **one-hot encoded** — 21
channels, a 1 in the channel of the residue's amino acid — and the output has
3 scores at every position.

| | Example 7 |
| --- | --- |
| input | a one-hot sequence, shape `(21, 700)` |
| output | 3 scores per position, shape `(3, 700)` |
| last layer | 1 × 1 convolution, 3 channels |
| loss | cross-entropy, padding ignored |
| metric | accuracy over the real residues of the test proteins |
| baseline | the most frequent structure; the most frequent structure of each amino acid (a lookup table, no context) |

**On disk:** every protein is one file `protein/<split>/00042.npz` holding two
arrays, padded to 700 positions: `aa`, the amino acid of every position (0–20,
21 = padding), and `ss`, its structure (0 = `H`, 1 = `C`, 2 = `E`, −1 =
padding). The padding is not a protein, so it must not count: its positions
are all zeros in the one-hot input, `nn.CrossEntropyLoss(ignore_index=-1)`
skips them, and so does the metric.

In [ ]:
structures = ["H", "C", "E"]   # the classes 0, 1, 2 of the array ss (as the data were prepared); -1 = padding


class ProteinDataset(Dataset):
    def __init__(self, split):
        self.files = sorted((DATA / "protein" / split).glob("*.npz"))   # one protein per file: arrays aa and ss

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        with np.load(self.files[i]) as protein:
            seq = torch.tensor(protein["aa"], dtype=torch.long)            # (700,) amino acids 0-20, padding 21
            target = torch.tensor(protein["ss"], dtype=torch.long)         # (700,)
        x = F.one_hot(seq, 22)[:, :21].T.float()                          # (21, 700): one-hot of 22 classes, class 21 (padding) dropped -> all zeros
        return x, target


protein_sets = {s: ProteinDataset(s) for s in SPLITS}
for s in SPLITS:
    ss = torch.stack([y for _, y in protein_sets[s]])
    real = ss >= 0
    print(f"{s:5s} {len(ss)} proteins, {real.sum(dim=1).float().mean():.0f} residues on average,",
          "structure H / C / E:", np.round(np.bincount(ss[real], minlength=3) / real.sum().item(), 3))
print("one protein:", tuple(protein_sets["train"][0][0].shape), "->", tuple(protein_sets["train"][0][1].shape))

torch.manual_seed(0)
protein_net = train(UNet(dim=1, c_in=21, c_out=3, widths=[32, 64, 128]),
                    protein_sets["train"], protein_sets["valid"], nn.CrossEntropyLoss(ignore_index=-1), epochs=12)

In [ ]:
scores, target = predict(protein_net, protein_sets["test"])
real = target >= 0                                       # the real residues, not the padding
accuracy = (scores.argmax(dim=1) == target)[real].float().mean().item()

aa_train = torch.stack([x.argmax(dim=0) for x, _ in protein_sets["train"]])   # back from one-hot to indices (padding becomes 0, masked out below)
ss_train = torch.stack([y for _, y in protein_sets["train"]])
aa_test = torch.stack([x.argmax(dim=0) for x, _ in protein_sets["test"]])
train_real = ss_train >= 0
majority = np.bincount(ss_train[train_real]).argmax()
lookup = torch.zeros(21, dtype=torch.long)              # the most frequent structure of each amino acid
for a in range(21):
    lookup[a] = np.bincount(ss_train[train_real & (aa_train == a)], minlength=3).argmax()
acc_majority = (target[real] == majority).float().mean().item()
acc_lookup = (lookup[aa_test[real]] == target[real]).float().mean().item()
print(f"test accuracy per residue   always '{structures[majority]}' {acc_majority:.3f}   "
      f"lookup by amino acid {acc_lookup:.3f}   U-Net {accuracy:.3f}")
results["7 Protein structure"] = dict(metric="accuracy", model=accuracy, baseline=acc_lookup)

i = 0                                                    # test protein 0, its real residues only
length = int(real[i].sum())
fig, ax = plt.subplots(figsize=(12, 1.8))
ax.imshow(np.stack([target[i, :length].numpy(), scores[i].argmax(dim=0)[:length].numpy()]), cmap="viridis",
          vmin=0, vmax=2, aspect="auto", interpolation="nearest")
ax.set_yticks([0, 1], ["true", "U-Net"])
ax.set_xlabel(f"residue (all {length}); colours: H = purple, C = teal, E = yellow")
plt.show()

### Summary — the same recipe seven times

Every example answered the same six questions — input, output, last layer,
loss, metric, baseline (the rows of its table); only the answers changed.
The table collects the test results against the baselines.

In [ ]:
summary = pd.DataFrame.from_dict(results, orient="index")
# MSE: lower is better; all the other metrics: higher is better
summary["better than the baseline"] = np.where(summary["metric"] == "MSE",
                                               summary["model"] < summary["baseline"],
                                               summary["model"] > summary["baseline"])
summary.round(4)

What to take away:

- **The output decides the head**: one answer per input → pooling + linear
  layer; one answer per position → U-Net + 1 × 1 convolution. 1D or 2D is
  only `Conv1d` or `Conv2d`.
- **The kind of answer decides the loss**: classes → cross-entropy (the
  network outputs logits, the softmax is in the loss); real numbers → MSE
  with no activation; masks → cross-entropy + Dice.
- **A variable-length answer** (QRS positions) is turned into a fixed-size
  one (a heatmap) and post-processed back.
- **What must not count is masked**: padding gets the target −1,
  `CrossEntropyLoss(ignore_index=-1)` skips it, and so does the metric.
- **The metric is not the loss**, and it is judged on the test set against
  a baseline that is computed, not guessed. Every choice — a threshold, a
  filter width, a baseline variant — is made on the validation set, never on
  the test set.

## Part 2 — Applications: you specify, the agent writes the code

Seven new problems. Each is one of the types of Part 1 on new data, with
something that makes it more than a copy: a label that is missing where you
would not expect it, a model that has to run on your phone, a metric that
is not the loss. **Pick three of the seven.** For each of them you write the
specification, your AI agent writes the code, and you check that the result
is right.

| # | Task | Type | Data |
| --- | --- | --- | --- |
| 1 | ECG delineation: where are P, QRS and T? | 1D, a class for every sample | LUDB: 200 patients, 10-s ECGs |
| 2 | Activity from the phone's accelerometer | 1D, a class for every sample — then on your phone | UCI HAPT: 30 people, phone on the waist |
| 3 | Calcium imaging → spikes | 1D signal-to-signal regression | spikefinder: 174 neurons |
| 4 | Counting cells | 2D, a density map → a count | synthetic fluorescence microscopy (VGG cells) |
| 5 | Nucleus segmentation and counting | 2D segmentation → separate objects | DSB 2018 nuclei |
| 6 | Colorization of photos | 2D image-to-image: regression or classes | Imagenette photos |
| 7 | Deblurring blurry text | 2D image-to-image regression, 2× upscaling | generated text images |

Every task has a cell that downloads, loads and shows its data, with a fixed
split into training, validation and test; keep it as it is, so that your
numbers can be compared with the baseline and with your classmates'. The
tasks do not need the data of Part 1, nor each other.

### How to work with the agent

1. **Run** the imports, the networks (`Encoder`, `Classifier`, `UNet`) and the
   `train`/`predict` cell of Part 1 — for Task 7 also the `Residual` cell of
   example 4 — then the data cell of your task. Read the shapes, look at the plot.
2. **Fill in the specification table** under the task yourself, before any
   code (the shapes are those of one sample, without the batch dimension). It
   is the core of your prompt, and the part the agent cannot decide for you.
3. **Ask for the datasets.** Look at one input–target pair.
4. **Ask for the network and a short training run** (a few steps). Do the
   network output and target shapes match your specification and loss? Is
   the loss finite, does it go down?
5. **Train, and make every choice on the validation set.**
6. **Evaluate on the test set** — your model and the baseline, with the
   metric of your table, not the training loss — and produce *What to show*.
   Look at the predictions, including the worst ones: most mistakes of an
   agent are found on a plot, not in a number.

For a GPU: *Runtime → Change runtime type → T4 GPU* in Colab, or let your
local agent send the training to a Colab GPU with the
[Colab GPU guide](https://tomasvicar.github.io/MLR-public/materials/colab-gpu.html)
of exercise 8. A starting point for a prompt:

> I am solving Task 1 of the notebook ex09_student.ipynb (attached). My specification: … Use the data cell of Task 1 and the functions `train`, `predict` and `UNet` of Part 1 without changing them. For now, write only the code that builds the training and validation datasets, print their shapes, and stop.

The numbers below are reference results, not guaranteed scores; report your result and explain if you do not beat the baseline.

The three tasks are the **compulsory hand-in** of this lab — see *Hand-in* at the end of the notebook. "What to show" under each task lists what it must contain.

### Task 1 — ECG delineation: where are P, QRS and T? (1D, a class for every sample)

Example 6 found only the QRS complexes. A cardiologist reads **intervals**
— the PR interval, the width of the QRS, the QT interval — and needs where
every wave **starts and ends**: the P wave (the atria contract), the QRS
complex (the ventricles) and the T wave (the ventricles recover). The
answer is one of four classes — background, P, QRS, T — for **every
sample**.

**Data:** the Lobachevsky University Electrocardiography Database
([LUDB](https://physionet.org/content/ludb/1.0.1/) on PhysioNet, Kalyakulina
et al. 2020): 200 patients, 10 s of a 12-lead ECG at
500 Hz — one sample is 2 ms — with every wave outlined by cardiologists. We
use lead II. The cell downloads the database (15 MB) and reads it with the
`wfdb` package (in Colab the cell installs it). It gives

- `ludb_x` — `(200, 5000)`, lead II of every patient, standardized per record;
- `ludb_y` — `(200, 5000)`, the class of every sample: 0 background, 1 P, 2 QRS, 3 T;
- `ludb_waves` — for every record, the list of its waves as `(class, onset, peak, offset)` in samples;
- `ludb_rhythm` — the rhythm of every patient (sinus rhythm, atrial fibrillation, …);
- `ludb_split` — the record indices of the 140 training, 30 validation and 30 test patients.

Look at the plot closely, also at the start and the end of the record.

In [ ]:
if "google.colab" in sys.modules:
    %pip install -q wfdb
import wfdb

LUDB_URL = ("https://physionet.org/static/published-projects/ludb/"
            "lobachevsky-university-electrocardiography-database-1.0.1.zip")
LUDB = Path("data/ludb/lobachevsky-university-electrocardiography-database-1.0.1")
if len(list((LUDB / "data").glob("*.hea"))) != 200 or not (LUDB / "ludb.csv").exists():
    os.makedirs("data/ludb", exist_ok=True)
    urllib.request.urlretrieve(LUDB_URL, "data/ludb.zip")       # 15 MB
    zipfile.ZipFile("data/ludb.zip").extractall("data/ludb")
    os.remove("data/ludb.zip")

FS = 500                                       # Hz: one sample = 2 ms
WAVES = ["background", "P", "QRS", "T"]
wave_code = {"p": 1, "N": 2, "t": 3}           # the wfdb symbols of the P wave, the QRS complex and the T wave
ludb_x, ludb_y, ludb_waves = [], [], []
for rec in range(1, 201):
    path = str(LUDB / "data" / str(rec))
    x = wfdb.rdrecord(path, channel_names=["ii"]).p_signal[:, 0]      # lead II, in mV
    ann = wfdb.rdann(path, "ii")                                     # the annotation of lead II
    y, waves = np.zeros(len(x), dtype=np.int64), []
    for k in range(1, len(ann.symbol) - 1):    # a wave is a triplet: "(" at the onset, its symbol at the peak, ")" at the offset
        if ann.symbol[k] in wave_code and ann.symbol[k - 1] == "(" and ann.symbol[k + 1] == ")":
            c, onset, peak, offset = wave_code[ann.symbol[k]], ann.sample[k - 1], ann.sample[k], ann.sample[k + 1]
            y[onset:offset + 1] = c
            waves.append((c, onset, peak, offset))
    ludb_x.append((x - np.median(x)) / (x.std() + 1e-6))
    ludb_y.append(y)
    ludb_waves.append(waves)
ludb_x, ludb_y = np.stack(ludb_x).astype(np.float32), np.stack(ludb_y)
meta = pd.read_csv(LUDB / "ludb.csv")
rhythm_of = dict(zip(meta["ID"], meta["Rhythms"]))
ludb_rhythm = [str(rhythm_of[rec]).strip().splitlines()[0] for rec in range(1, 201)]
perm = np.random.default_rng(0).permutation(200)                    # one record = one patient
ludb_split = {"train": perm[:140], "valid": perm[140:170], "test": perm[170:]}
print("signals", ludb_x.shape, "labels", ludb_y.shape, {s: len(v) for s, v in ludb_split.items()})
print("fraction of the samples in each class:", dict(zip(WAVES, np.round(np.bincount(ludb_y.ravel()) / ludb_y.size, 3))))

WAVE_COLORS = {1: "tab:green", 2: "tab:red", 3: "tab:blue"}
i = ludb_split["train"][0]
t = np.arange(ludb_x.shape[1]) / FS
plt.figure(figsize=(12, 2.8))
plt.plot(t, ludb_x[i], "k", linewidth=0.7)
for c, color in WAVE_COLORS.items():
    plt.fill_between(t, ludb_x[i].min(), ludb_x[i].max(), where=ludb_y[i] == c, color=color, alpha=0.3,
                     linewidth=0, label=WAVES[c])
plt.xlim(0, 10)
plt.xlabel("time (s)")
plt.title(f"record {i + 1}, lead II ({ludb_rhythm[i]}), with the cardiologists' labels", fontsize=10)
plt.legend(ncol=3, fontsize=8, loc="upper right")
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** The baseline is a classical delineator: find the R
peaks (the tops of the QRS) with `find_peaks` on the band-passed signal, and
put the P wave, the QRS and the T wave at their median distances from the R
peak, measured on the training patients. On the test patients of this split
it reaches a Dice of about **0.74 for P, 0.89 for QRS and 0.85 for T**. Your
1D U-Net should beat it on all three waves.

**What to show:**

- your specification table;
- the test Dice of every wave, next to the baseline's;
- one test record with the true and the predicted waves, one above the other;
- the QRS duration in ms of one test patient, predicted against annotated.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 2 — Activity from the phone's accelerometer (1D, a class for every sample)

Every phone has an accelerometer, and a fitness app tells walking from
climbing stairs from sitting. Here the network labels **every sample** of
the acceleration with one of six activities or a transition between them
— the protein example of Part 1, on a body signal. Then **your own model runs
on your phone**: the course app reads the accelerometer in the browser, runs
your network on it and draws a colour-coded timeline of what you did.

**Data:** UCI HAPT, *Smartphone-Based Recognition of Human Activities and
Postural Transitions* (Reyes-Ortiz, Oneto, Samà, Parra, Anguita;
[UCI repository](https://archive.ics.uci.edu/dataset/341)):
30 people with a smartphone on the waist, the 3-axis acceleration in **g**
at **50 Hz**, 61 recordings of a few minutes. Every sample is labelled
walking, upstairs, downstairs, sitting, standing, lying, or one of six
postural transitions (sit-to-stand, …, merged here into one class
`transition`); the samples between the labelled activities have the label
−1. The cell downloads the 80 MB zip and keeps only the accelerometer
files. It gives

- `HAPT_CLASSES` — the seven class names, `HAPT_COLORS` — their colours in the app;
- `hapt` — every recording as `(experiment, user, acceleration, labels)`, the acceleration of shape `(T, 3)` in g, the labels `(T,)` with values 0–6 or −1;
- `hapt_split` — the recordings of users 1–20 (training), 21–25 (validation) and 26–30 (test): a split by person.

In [ ]:
HAPT_URL = ("https://archive.ics.uci.edu/static/public/341/"
            "smartphone+based+recognition+of+human+activities+and+postural+transitions.zip")
HAPT = Path("data/hapt")
if len(list(HAPT.glob("acc_exp*_user*.txt"))) != 61 or not (HAPT / "labels.txt").exists():
    os.makedirs(HAPT, exist_ok=True)
    urllib.request.urlretrieve(HAPT_URL, "data/hapt.zip")       # 80 MB
    with zipfile.ZipFile("data/hapt.zip") as z:                 # only the accelerometer and the labels
        for name in z.namelist():
            if name.startswith("RawData/acc_") or name == "RawData/labels.txt":
                (HAPT / Path(name).name).write_bytes(z.read(name))
    os.remove("data/hapt.zip")

HAPT_CLASSES = ["walking", "upstairs", "downstairs", "sitting", "standing", "lying", "transition"]
HAPT_COLORS = ["#2a9d8f", "#e76f51", "#f4a261", "#457b9d", "#8ab17d", "#9b5de5", "#adb5bd"]
labels_txt = np.loadtxt(HAPT / "labels.txt", dtype=int)       # experiment, user, activity 1-12, first and last sample (from 1)
hapt = []
for f in sorted(HAPT.glob("acc_exp*_user*.txt")):
    exp, user = int(f.stem[7:9]), int(f.stem[-2:])
    acc = np.loadtxt(f, dtype=np.float32)                      # (T, 3): x, y, z in g
    lab = np.full(len(acc), -1)                                # -1: between the labelled activities
    for _, _, activity, first, last in labels_txt[labels_txt[:, 0] == exp]:
        lab[first - 1:last] = activity - 1 if activity <= 6 else 6     # the six transitions -> one class
    hapt.append((exp, user, acc, lab))
users = {"train": range(1, 21), "valid": range(21, 26), "test": range(26, 31)}
hapt_split = {s: [r for r in hapt if r[1] in u] for s, u in users.items()}
for s, recs in hapt_split.items():
    lab = np.concatenate([r[3] for r in recs])
    print(f"{s:5s} {len(recs)} recordings, {len(lab)} samples; labels -1, 0-6:", np.bincount(lab + 1))

exp, user, acc, lab = hapt_split["train"][0]
t = np.arange(len(acc)) / 50
fig, axes = plt.subplots(2, 1, figsize=(12, 3.4), sharex=True, gridspec_kw={"height_ratios": [3, 0.5]})
for k, axis in enumerate("xyz"):
    axes[0].plot(t, acc[:, k], linewidth=0.6, label=axis)
axes[0].set_ylabel("acceleration (g)")
axes[0].legend(ncol=3, fontsize=8, loc="upper right")
axes[0].set_title(f"HAPT experiment {exp}, user {user}", fontsize=10)
axes[1].imshow(np.ma.masked_less(lab[None], 0), aspect="auto", cmap=plt.matplotlib.colors.ListedColormap(HAPT_COLORS),
               vmin=-0.5, vmax=6.5, extent=[0, t[-1], 0, 1], interpolation="nearest")
axes[1].set_yticks([])
axes[1].set_xlabel("time (s); white = no label")
fig.legend(handles=[plt.matplotlib.patches.Patch(color=c, label=n) for n, c in zip(HAPT_CLASSES, HAPT_COLORS)],
           loc="lower center", ncol=7, fontsize=8, frameon=False)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** Always answering the most frequent class of the
training set (*standing*) gives a test accuracy of about 0.18 and a
**macro F1 of about 0.04** (macro F1 is the mean of the F1 scores of the
seven classes, so a rare class counts as much as a frequent one). A 1D
U-Net reaches a macro F1 **around 0.9** on the test users.

**Your model on your phone.**

1. Export the trained network to ONNX, as in exercise 8: input
   `1 × 3 × window` (the axes x, y, z), output `1 × 7 × window` logits, with
   the metadata `model_info` that tells the app how to feed it — the class
   names, the units the network was trained on (`"g"` or `"m/s2"`; the
   browser gives m/s²), the sampling rate in Hz, the window length and the
   axis order. The section *How to export your model* of the app shows the code.
2. Open the [activity app of the course](https://tomasvicar.github.io/MLR-public/apps/activity/)
   on your phone, load your `.onnx` file, tap *Start* and put the phone on your waist.
3. Do a short script — stand 20 s, walk 30 s, sit 20 s, take the stairs, lie
   down on a bench — then tap *Stop*: the app draws the timeline and lets you
   download it as CSV.

*Simulate* replays a recording of a test user, so the app can be tried on a laptop.

**What to show:**

- your specification table;
- the macro F1 and the per-class F1 on the test users, the confusion matrix, next to the baseline;
- the true and the predicted timeline of one test recording;
- the timeline of your own recording from the app, next to the script you did, and one or two sentences on where it fails and why.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 3 — Calcium imaging → spikes (1D signal-to-signal regression)

A neuron communicates with **spikes**, electrical pulses about 1 ms long.
With a microscope, neuroscientists record hundreds of neurons at once
through a **calcium indicator**: a dye that glows when calcium enters the
neuron after a spike. The fluorescence rises within tens of milliseconds
and decays over hundreds, so the recording is a slow, noisy, smeared
version of the spike train. Getting the spikes back is a deconvolution — a
1D U-Net that maps the fluorescence to a **spike rate**, sample by sample.

**Data:** the training part of the **spikefinder** challenge (Berens et al.
2018, [PLoS Comput Biol 14: e1006157](https://doi.org/10.1371/journal.pcbi.1006157);
[github.com/codeneuro/spikefinder](https://github.com/codeneuro/spikefinder)),
in which convolutional networks won: 174 neurons in 10 datasets (different
labs and indicators), each with the fluorescence and, recorded electrically
at the same time, the true spikes; everything at 100 Hz. The cell
downloads it from the challenge's own address (35 MB). It gives

- `sf_calcium[ds]`, `sf_spikes[ds]` for the datasets `ds` = 1 … 10 — tables
  with one column per neuron: the fluorescence, and the number of spikes in
  every 10-ms sample. Recordings differ in length; a column ends with NaN
  where its recording is over;
- `sf_split` — the neurons of the training, validation and test set as
  `(dataset, column)` pairs, split by neuron inside every dataset.

The **metric** of the challenge: the predicted rate and the true spike
counts are both summed into bins of 40 ms (25 Hz), and their Pearson
correlation is computed for every test neuron; the score is the mean over
the neurons.

In [ ]:
SF_URL = "https://s3.amazonaws.com/neuro.datasets/challenges/spikefinder/spikefinder.train.zip"
SF = Path("data/spikefinder/spikefinder.train")
if len(list(SF.glob("*.train.*.csv"))) != 20:
    os.makedirs("data/spikefinder", exist_ok=True)
    urllib.request.urlretrieve(SF_URL, "data/spikefinder.zip")      # 35 MB
    zipfile.ZipFile("data/spikefinder.zip").extractall("data/spikefinder")
    os.remove("data/spikefinder.zip")

SF_FS = 100                                                         # Hz
sf_calcium = {ds: pd.read_csv(SF / f"{ds}.train.calcium.csv") for ds in range(1, 11)}
sf_spikes = {ds: pd.read_csv(SF / f"{ds}.train.spikes.csv") for ds in range(1, 11)}
neurons = [(ds, j) for ds in range(1, 11) for j in range(sf_calcium[ds].shape[1])]
rng = np.random.default_rng(0)
sf_split = {"train": [], "valid": [], "test": []}                 # split by neuron inside every dataset
for ds in range(1, 11):
    idx = rng.permutation([k for k, n in enumerate(neurons) if n[0] == ds])
    n_test, n_val = max(1, round(0.2 * len(idx))), max(1, round(0.1 * len(idx)))
    sf_split["test"] += [neurons[k] for k in idx[:n_test]]
    sf_split["valid"] += [neurons[k] for k in idx[n_test:n_test + n_val]]
    sf_split["train"] += [neurons[k] for k in idx[n_test + n_val:]]
hours = sum(sf_calcium[ds].iloc[:, j].notna().sum() for ds, j in neurons) / SF_FS / 3600
print(f"{len(neurons)} neurons, {hours:.1f} hours;", {s: len(v) for s, v in sf_split.items()})

ds, j = sf_split["train"][0]
calcium, spikes = sf_calcium[ds].iloc[:, j].to_numpy(), sf_spikes[ds].iloc[:, j].to_numpy()
part = slice(30 * SF_FS, 60 * SF_FS)
t = np.arange(len(calcium)) / SF_FS
fig, axes = plt.subplots(2, 1, figsize=(12, 3.6), sharex=True)
axes[0].plot(t[part], calcium[part], color="tab:green", linewidth=0.8)
axes[0].set_ylabel("fluorescence")
axes[0].set_title(f"dataset {ds}, neuron {j}: 30 s of {np.isfinite(calcium).sum() / SF_FS:.0f} s", fontsize=10)
axes[1].vlines(t[part][spikes[part] > 0], 0, spikes[part][spikes[part] > 0], color="k", linewidth=1)
axes[1].set_ylabel("spikes per 10 ms")
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** Two classical guesses: the fluorescence itself as
the rate (correlation about 0.21 on the test neurons), and the **positive
part of its derivative** after smoothing — the fluorescence jumps up right
after a spike — with the smoothing width chosen on the validation neurons
(correlation about **0.45**). Your network should beat the derivative, on
average and in most of the ten datasets.

**What to show:**

- your specification table;
- the mean test correlation of your network and of both baselines, and the same per dataset;
- 30 s of one test neuron: the fluorescence, the true spikes and your predicted rate.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 4 — Counting cells (2D, a density map → a count)

"How many cells are in this image?" is one of the most frequent questions
in a biology lab. Cells touch and overlap, so a threshold and counting the
blobs misses many of them. Example 6 turned every QRS into a bump of a
heatmap and then found the peaks; here the same idea goes one step
further: put a small Gaussian on the centre of every annotated cell, and
the image of the bumps — a **density map** — sums to the number of cells.
A U-Net learns to paint the density map, and the count is the **sum of its
output**: nobody has to find the individual cells.

**Data:** synthetic fluorescence-microscopy images of cells (made with the
SIMCEP simulator) from the counting paper of the Oxford Visual Geometry
Group (Lempitsky & Zisserman, *Learning to count objects in images*, NIPS
2010; [the dataset page](https://www.robots.ox.ac.uk/~vgg/research/counting/index_org.html)):
200 images of 256 × 256 pixels with a dot at the centre of every cell. The
cell downloads it from the VGG page (16 MB). It gives

- `cell_img` — `(200, 256, 256)`, the blue channel of every image, in [0, 1] (the red and green channels are empty);
- `cell_dots` — `(200, 256, 256)`, 1 at the centre of every cell, 0 elsewhere;
- `cell_count` — `(200,)`, the number of cells in every image (the sum of its dots);
- `cell_split` — the image indices of the training (images 1–80), validation (81–100) and test set (101–200, the test set of the paper).

In [ ]:
CELLS_URL = "https://www.robots.ox.ac.uk/~vgg/research/counting/cells.zip"
CELLS = Path("data/cells")
if len(list(CELLS.glob("*.png"))) != 400:                       # 200 images + 200 dot images
    os.makedirs(CELLS, exist_ok=True)
    urllib.request.urlretrieve(CELLS_URL, "data/cells.zip")     # 16 MB
    zipfile.ZipFile("data/cells.zip").extractall(CELLS)
    os.remove("data/cells.zip")

cell_img, cell_dots = [], []
for i in range(1, 201):
    rgb = np.asarray(Image.open(CELLS / f"{i:03d}cell.png").convert("RGB"), dtype=np.float32) / 255
    dots = np.asarray(Image.open(CELLS / f"{i:03d}dots.png").convert("RGB"))
    cell_img.append(rgb[..., 2])                                # the cells are in the blue channel
    cell_dots.append((dots[..., 0] > 0) & (dots[..., 1] == 0))  # a red dot at every cell centre
cell_img = np.stack(cell_img)
cell_dots = np.stack(cell_dots).astype(np.float32)
cell_count = cell_dots.sum(axis=(1, 2))
cell_split = {"train": np.arange(0, 80), "valid": np.arange(80, 100), "test": np.arange(100, 200)}
print("images", cell_img.shape, "| cells per image", int(cell_count.min()), "–", int(cell_count.max()),
      f"(mean {cell_count.mean():.0f})", "|", {s: len(v) for s, v in cell_split.items()})

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, i in zip(axes, [0, 1, int(np.argmax(cell_count[:80]))]):
    ax.imshow(cell_img[i], cmap="gray")
    rows, cols = np.nonzero(cell_dots[i])
    ax.plot(cols, rows, "r.", markersize=2)
    ax.set_title(f"image {i + 1}: {cell_count[i]:.0f} cells (red: the dots)", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** Two baselines: answering the mean count of the
training images for every image (a count MAE of about **56** cells
on the test images), and Otsu's threshold followed by counting the
connected blobs (MAE about **95**: touching cells merge into one
blob, so it finds only about 45 % of them). Your network should
count within a few cells — a count MAE well under 10 in most runs, out of
about 170.

**What to show:**

- your specification table;
- the check that your target sums to the count, on a few training images;
- the test count MAE of your network and of both baselines;
- the predicted against the true count of all test images (a scatter plot with the diagonal);
- one test image with its target and its predicted density map, each with its sum.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 5 — Nucleus segmentation and counting (2D segmentation → separate objects)

Example 2 segmented one lesion per photo: a mask was all we needed. A
biologist looking at a fluorescence image of cell nuclei needs more — the
**individual nuclei**: how many there are, how large each one is. Two
nuclei that touch are one blob in a binary mask, so a perfect foreground
mask still counts them as one. The usual trick is a third class: every
pixel is **background, the interior of a nucleus, or its boundary**; the
predicted boundaries separate touching nuclei, and the connected regions
of the interior are the nuclei.

**Data:** the nuclei of the 2018 Data Science Bowl
([BBBC038](https://bbbc.broadinstitute.org/BBBC038)), the subset with
corrected annotations used by the
[StarDist paper](https://arxiv.org/abs/1806.03535). Fluorescence
images of many cell types and sizes (256 × 256 up to 520 × 696 pixels),
each with an image of the **instance labels**: 0 for the background,
1, 2, 3, … for the separate nuclei. The cell downloads the StarDist
release (27 MB). It gives

- `nuc_img[s]` — for `s` in `"train"`, `"valid"`, `"test"` a list of gray images, float in [0, 1] (normalized per image: the 1st and the 99.8th percentile → 0 and 1); sizes differ;
- `nuc_lab[s]` — the list of the matching label images (int, 0 = background, k = nucleus k);
- the split: the 447 training images of the release divided into 402 training and 45 validation images (random, seed 0), and the release's 50 test images.

In [ ]:
import tifffile

NUCLEI_URL = "https://github.com/stardist/stardist/releases/download/0.1.0/dsb2018.zip"
NUCLEI = Path("data/dsb2018")
if len(list(NUCLEI.glob("*/*/*.tif"))) != 2 * (447 + 50):        # images and masks, train and test
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(NUCLEI_URL, "data/dsb2018.zip")   # 27 MB
    zipfile.ZipFile("data/dsb2018.zip").extractall("data")
    os.remove("data/dsb2018.zip")


images, labels = {}, {}
for part in ["train", "test"]:                                   # the two folders of the release
    images[part], labels[part] = [], []
    for path in sorted((NUCLEI / part / "images").glob("*.tif")):
        img = tifffile.imread(path).astype(np.float32)
        low, high = np.percentile(img, (1, 99.8))
        images[part].append(np.clip((img - low) / max(high - low, 1e-6), 0, 1).astype(np.float32))
        labels[part].append(tifffile.imread(NUCLEI / part / "masks" / path.name).astype(np.int64))
order = np.random.default_rng(0).permutation(len(images["train"]))
nuc_img = {"train": [images["train"][k] for k in order[45:]], "valid": [images["train"][k] for k in order[:45]],
           "test": images["test"]}
nuc_lab = {"train": [labels["train"][k] for k in order[45:]], "valid": [labels["train"][k] for k in order[:45]],
           "test": labels["test"]}
for s in ["train", "valid", "test"]:
    print(f"{s:5s}: {len(nuc_img[s])} images, {sum(len(np.unique(l)) - 1 for l in nuc_lab[s])} nuclei")


def show_labels(ax, lab):
    """Every nucleus in its own random colour, the background black."""
    colors = np.random.default_rng(1).uniform(0.25, 1, (lab.max() + 1, 3))
    colors[0] = 0
    ax.imshow(colors[lab])


fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, k in enumerate([0, 3, 7, 12]):
    axes[0, col].imshow(nuc_img["train"][k], cmap="gray")
    axes[0, col].set_title(f"training image {k}, {nuc_img['train'][k].shape}", fontsize=9)
    show_labels(axes[1, col], nuc_lab["train"][k])
    axes[1, col].set_title(f"{len(np.unique(nuc_lab['train'][k])) - 1} nuclei", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** Otsu's threshold already gives a foreground
**Dice of about 0.90** on the test images — on Dice alone there is
little left to win. Count its connected blobs as nuclei, though, and the
**object-level F1** (a true nucleus counts as found when one predicted
nucleus overlaps it with an IoU above 0.5) is about **0.78**, and even
the *true* foreground mask cut into connected blobs reaches only about
0.90, because touching nuclei merge. Your network should beat Otsu on
the object F1. Through its boundary class it should also find more nuclei
correctly than by cutting its own foreground mask into blobs, and usually
count them better. The **count error** of an image is |counted − true| / true;
report its mean over the test images.

**What to show:**

- your specification table;
- the target of one training image with touching nuclei (it must separate them);
- the foreground Dice, the object F1 at IoU 0.5 and the count error on the test images, for your network and for Otsu;
- the predicted against the true number of nuclei per test image;
- two or three test images: the true nuclei, Otsu's and yours, each nucleus in its own colour.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 6 — Colorization of photos (2D image-to-image: regression or classes)

Give the network the black-and-white version of a photo and let it paint
the colours back. Every photo is its own training pair — no annotation is
needed — and the **luminance** (the black-and-white image) is kept as it
is: the network predicts only the two **colour channels**. The question of
this task is the last layer. Predicting the colour as two numbers with an
L1 or MSE loss sounds natural; but a car may be red or blue, and the
colour that minimizes the average error over all possible answers is a
dull brownish gray. The alternative (Zhang et al. 2016,
[Colorful Image Colorization](https://arxiv.org/abs/1603.08511)) cuts the
colour plane into a few hundred **bins** and treats every pixel as a
classification into them — with its own trap, as you will find out.

**Data:** [Imagenette](https://github.com/fastai/imagenette) (fast.ai,
Jeremy Howard): 10 easy classes of ImageNet (a church, a fish, a petrol
pump, a golf ball, …), the version with the short side of 160 pixels. The
cell downloads it (95 MB) and makes every photo 64 × 64 (the short side resized,
the centre cut out). It gives

- `col_rgb[s]` — for `s` in `"train"`, `"valid"`, `"test"`, the photos as `uint8` arrays `(n, 64, 64, 3)`;
- the split: all 9469 training photos of Imagenette for training; 100 validation and 500 test photos from its validation part (random, seed 0).

**Use a GPU** (*Runtime → Change runtime type → T4 GPU* in Colab, or from
your local agent with the [Colab GPU guide](https://tomasvicar.github.io/MLR-public/materials/colab-gpu.html)): one
network trains for about 6 minutes on a T4. A useful colour space is
JPEG's YCbCr — Y is the luminance, Cb and Cr the colour — or Lab
(`skimage.color.rgb2lab`).

In [ ]:
import tarfile

COL_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
COL = Path("data/imagenette2-160")
if len(list(COL.glob("*/*/*.JPEG"))) != 9469 + 3925:
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(COL_URL, "data/imagenette2-160.tgz")   # 95 MB
    with tarfile.open("data/imagenette2-160.tgz") as archive:
        archive.extractall("data")
    os.remove("data/imagenette2-160.tgz")


def load_photo(path, size=64):
    """The short side resized to 64 pixels, then the central 64 x 64 square."""
    img = Image.open(path).convert("RGB")
    scale = size / min(img.size)
    img = img.resize((max(size, round(img.width * scale)), max(size, round(img.height * scale))), Image.BICUBIC)
    left, top = (img.width - size) // 2, (img.height - size) // 2
    return np.asarray(img.crop((left, top, left + size, top + size)))


train_files = sorted((COL / "train").glob("*/*.JPEG"))
val_files = sorted((COL / "val").glob("*/*.JPEG"))
order = np.random.default_rng(0).permutation(len(val_files))
col_files = {"train": train_files, "valid": [val_files[k] for k in order[500:600]], "test": [val_files[k] for k in order[:500]]}
col_rgb = {s: np.stack([load_photo(f) for f in files]) for s, files in col_files.items()}
print({s: v.shape for s, v in col_rgb.items()})

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for k in range(8):
    axes[0, k].imshow(col_rgb["test"][k])
    axes[1, k].imshow(col_rgb["test"][k] @ np.array([0.299, 0.587, 0.114]), cmap="gray", vmin=0, vmax=255)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
axes[0, 0].set_title("the photo (the target)", fontsize=9, loc="left")
axes[1, 0].set_title("its luminance (the input)", fontsize=9, loc="left")
plt.tight_layout()
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** The gray image — the input itself, without any
colour — has an RGB **PSNR of about 25 dB** against the true photos
and a colourfulness of 0 (the true photos: about 0.04). A
network trained with L1 on the two colour channels reaches a *lower* PSNR
than gray and colours almost nothing. The target of this task is not a
number: train the regression (A), the bins (B) and the rebalanced bins (B+),
and find a version whose photos are **visibly colourful** — then explain
what each metric says about it.

**What to show:**

- your specification table, for the regression (A) and for the bins (B, B+);
- the PSNR, the chroma L1 and the colourfulness on the test photos for every network and for the gray baseline;
- a grid of 8 test photos: the gray input, every network's colours, the truth;
- two sentences: which network you would pick, and what the PSNR would have picked.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

### Task 7 — Deblurring blurry text (2D image-to-image regression, 2× upscaling)

A photo of a page taken from too far and slightly out of focus: the
letters are small, blurred and noisy. Example 4 removed noise from photos
at their own resolution; here the network must also **double the
resolution** and make the letters sharp again — deblurring and
super-resolution at once. Text is a good case to see what a network
learns: the shapes of letters. It is also a good case to see what it does
*not* learn: letters of a font it has never seen.

**Data:** generated by the cell, nothing is downloaded. Lines of random
words (letters drawn with their English frequencies) are rendered with
Pillow, black on white, at 64 × 64 pixels; every patch is then degraded:
Gaussian blur (σ between 0.8 and 1.6 px), 2× downsampling (the mean of
2 × 2 pixels) and noise (σ = 0.01) — the 32 × 32 input. The fonts are the
ones shipped with matplotlib (DejaVu, STIX, Computer Modern), so they are
the same in Colab and on your computer. It gives

- `text_sharp[s]` — `(n, 64, 64)`, the sharp patches in [0, 1] (white = 1), and `text_low[s]` — `(n, 32, 32)`, their degraded versions, for `s` in
  `"train"` (6000 patches), `"valid"` and `"test"` (400 each, new words in the five training fonts) and `"unseen"` (400 patches in three fonts the training never sees);
- `TEXT_FONTS` — the font names of the training and of the unseen patches;
- `render_text(font, size, text=...)` and `degrade(sharp, sigma)` — the generator, for your own sentences.

In [ ]:
import matplotlib
from PIL import ImageDraw, ImageFont

FONT_DIR = Path(matplotlib.get_data_path()) / "fonts" / "ttf"      # fonts shipped with matplotlib
TEXT_FONTS = {"train": ["DejaVuSans", "DejaVuSans-Bold", "DejaVuSans-Oblique", "DejaVuSerif", "STIXGeneral"],
              "unseen": ["DejaVuSansMono", "STIXGeneralItalic", "cmss10"]}
LETTERS = list("abcdefghijklmnopqrstuvwxyz")
FREQ = np.array([8.2, 1.5, 2.8, 4.3, 12.7, 2.2, 2.0, 6.1, 7.0, 0.15, 0.77, 4.0, 2.4, 6.7, 7.5, 1.9, 0.1, 6.0, 6.3,
                 9.1, 2.8, 0.98, 2.4, 0.15, 2.0, 0.07])           # English letter frequencies (%)
text_rng = np.random.default_rng(0)


def random_word():
    word = "".join(text_rng.choice(LETTERS, text_rng.integers(2, 9), p=FREQ / FREQ.sum()))
    return word.capitalize() if text_rng.random() < 0.2 else word


def render_text(font, size, text=None):
    """Black text on white, float in [0, 1]: a 64 x 64 patch full of random words, or one given line
    (the image is then as wide as the line, rounded up to a multiple of 8)."""
    pil_font = ImageFont.truetype(str(FONT_DIR / f"{font}.ttf"), size)
    if text is None:
        width = height = 64
    else:
        left, top, right, bottom = pil_font.getbbox(text)
        width, height = -(-(right + 8) // 8) * 8, -(-(bottom + 8) // 8) * 8
    img = Image.new("L", (width, height), 255)
    draw = ImageDraw.Draw(img)
    if text is None:
        y = int(text_rng.integers(-size // 2, 2))
        while y < height:
            draw.text((int(text_rng.integers(-20, 2)), y), " ".join(random_word() for _ in range(6)), font=pil_font, fill=0)
            y += int(size * text_rng.uniform(1.1, 1.4))
    else:
        draw.text((4, 4), text, font=pil_font, fill=0)
    return np.asarray(img, dtype=np.float32) / 255


def degrade(sharp, sigma):
    """Gaussian blur, 2x downsampling (the mean of 2 x 2 pixels), noise 0.01."""
    blurred = gaussian_filter(sharp, sigma)
    h, w = sharp.shape
    small = blurred.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3))
    return np.clip(small + text_rng.normal(0, 0.01, small.shape), 0, 1).astype(np.float32)


text_sharp, text_low = {}, {}
for s, n, fonts in [("train", 6000, TEXT_FONTS["train"]), ("valid", 400, TEXT_FONTS["train"]),
                    ("test", 400, TEXT_FONTS["train"]), ("unseen", 400, TEXT_FONTS["unseen"])]:
    text_sharp[s] = np.stack([render_text(fonts[i % len(fonts)], int(text_rng.integers(13, 19))) for i in range(n)])
    text_low[s] = np.stack([degrade(img, text_rng.uniform(0.8, 1.6)) for img in text_sharp[s]])
print({s: (text_sharp[s].shape, text_low[s].shape) for s in text_sharp})

fig, axes = plt.subplots(2, 8, figsize=(12, 3.3))
for k, (s, i) in enumerate([("train", i) for i in range(5)] + [("unseen", i) for i in range(3)]):
    axes[0, k].imshow(text_sharp[s][i], cmap="gray", vmin=0, vmax=1)
    axes[0, k].set_title(f"{s}: {TEXT_FONTS['train' if s == 'train' else 'unseen'][i]}", fontsize=7)
    axes[1, k].imshow(text_low[s][i], cmap="gray", vmin=0, vmax=1)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
axes[1, 0].set_title("the input, 32 x 32", fontsize=7)
plt.tight_layout()
plt.show()

**Your specification** — fill it in before you ask the agent for any code:

| | your answer |
| --- | --- |
| network input (shape) | |
| network output (shape) | |
| last layer | |
| training loss | |
| evaluation metric | |
| baseline | |

**Baseline and target.** The classical answer is to upscale with bicubic
interpolation and sharpen with an **unsharp mask** (add a multiple of the
difference between the image and its blurred copy). On the test patches it
reaches a **PSNR of about 14.3 dB** (training fonts, new words) and
about 14.7 dB on the unseen fonts. Your network should be several dB
better on both — and readable on a whole sentence in an unseen font.

**What to show:**

- your specification table;
- the PSNR of your network and of the classical baselines, on the test patches and on the unseen fonts;
- a few test patches in the unseen fonts: the input, bicubic, your network, the truth;
- one sentence of your own in an unseen font (`render_text`, then `degrade`), restored by your network — the network is fully convolutional, so it works on the whole line.

*Your agent's code goes into the cells below (add as many as you need), or into a script of your own.*

## Hand-in

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit three tasks of Part 2 within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

Create a **private** GitHub repository `mlr-ex09` and share it with
**`MLRteacher`**: Settings → Collaborators → Add people. A pending invitation
is sufficient. The exercise page shows how your agent can do it with `gh`.

Include:

1. **The code of your three tasks**, one folder per task (e.g. `task1_ludb/`):
   your notebook with its outputs, or the scripts your agent wrote, so that
   the results can be reproduced from a fresh clone; a script saves the
   figures and the numbers it produces.
2. `README.md`: which three tasks you did, how to run each of them, and
   their dependencies.
3. **Everything "What to show" lists under each of your tasks** — the
   numbers next to the baseline and the figures (as PNG files saved by your
   scripts, or in the notebook outputs). For Task 2 also your `.onnx` model and the timeline of
   your own recording from the app.
4. **`REVIEW.md`, written by you:** for each task your specification table,
   what you checked (the smoke test, the baseline, the predictions you looked
   at) and what the agent got wrong and how you found out — or, if you
   found no error, which checks convinced you.

Use the validation split for every choice; the test split is for the final
result only. What is pushed before the deadline counts. Hand-ins are checked
with AI.

<p style="color: #c00000;"><strong>! Plagiarism check: all hand-ins will be checked for plagiarism.</strong></p>

The agent may write the code; your own review is your responsibility.